In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import optuna 
from statsmodels.stats.outliers_influence import variance_inflation_factor
from scipy import stats
import missingno as msno
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.feature_selection import mutual_info_classif
from sklearn.model_selection import cross_val_score, train_test_split, StratifiedKFold
from sklearn.pipeline import Pipeline
import statsmodels.api as sm
from sklearn.model_selection import learning_curve
import optuna
#from utils.perm_class import ClassificationCV
import seaborn as sns
from sklearn.metrics import roc_auc_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.linear_model import LogisticRegression

Initialize

In [ ]:
df = pd.read_csv('..\Data\loan_data_sample.csv')
features = ['loan_amnt', 'term', 'emp_length', 'home_ownership',
       'annual_inc', 'verification_status', 'purpose', 'dti', 'delinq_2yrs',
       'inq_last_6mths', 'mths_since_last_delinq', 'mths_since_last_record',
       'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 'total_acc',
       'initial_list_status', 'mths_since_last_major_derog',
       'application_type', 'tot_coll_amt', 'tot_cur_bal', 'open_acc_6m',
       'open_act_il', 'open_il_12m', 'open_il_24m', 'mths_since_rcnt_il',
       'total_bal_il', 'il_util', 'open_rv_12m', 'open_rv_24m', 'max_bal_bc',
       'all_util', 'total_rev_hi_lim', 'inq_fi', 'total_cu_tl', 'inq_last_12m',
       'acc_open_past_24mths', 'avg_cur_bal', 'bc_open_to_buy', 'bc_util',
       'chargeoff_within_12_mths', 'mo_sin_old_il_acct',
       'mo_sin_old_rev_tl_op', 'mo_sin_rcnt_rev_tl_op', 'mo_sin_rcnt_tl',
       'mort_acc', 'mths_since_recent_bc', 'mths_since_recent_bc_dlq',
       'mths_since_recent_inq', 'mths_since_recent_revol_delinq',
       'num_accts_ever_120_pd', 'num_actv_bc_tl', 'num_actv_rev_tl',
       'num_bc_sats', 'num_bc_tl', 'num_il_tl', 'num_op_rev_tl',
       'num_rev_accts', 'num_rev_tl_bal_gt_0', 'num_sats', 'num_tl_120dpd_2m',
       'num_tl_30dpd', 'num_tl_90g_dpd_24m', 'num_tl_op_past_12m',
       'pct_tl_nvr_dlq', 'percent_bc_gt_75', 'pub_rec_bankruptcies',
       'tax_liens', 'tot_hi_cred_lim', 'total_bal_ex_mort', 'total_bc_limit',
       'total_il_high_credit_limit',
       'months_sincefrst_credit', 'public_record', 'is_consolidation',
       'addr_state', 'is_currently_delinq', 'has_il_history']

index_sql = 'Loan_ID'
target = 'predictor'

df_features  = df[features]
df_predictor = pd.Series(df[target])


#Imputing col's we imputed with 999 in SQL
imputed_cols = [
    'mths_since_last_delinq', 'mths_since_last_record', 
    'mths_since_last_major_derog', 'mths_since_recent_bc_dlq', 
    'mths_since_recent_inq', 'mths_since_recent_revol_delinq'
]

for col in imputed_cols:
    df_features[col] = df_features[col].replace(999.0, np.nan)




X_train, X_test, y_train,y_test = train_test_split(df_features,df_predictor,stratify=df_predictor,test_size=.2,random_state=11)

categorical_features = X_train.select_dtypes(include=['object','category']).columns.tolist()
numerical_features = X_train.select_dtypes(include=['number']).columns.tolist()

Pipeline for Flags / missing values  --- Target Encoding

In [ ]:
from sklearn import set_config


categorical_features = ['home_ownership', 'verification_status', 'application_type', 'addr_state']

X_temp_data = X_train.copy()
X_temp_data['predictor'] = y_train
state_means = X_temp_data.groupby('addr_state')['predictor'].mean()
global_default_mean = y_train.mean()

m= 10 #smoothing parameter
state_means = X_temp_data.groupby('addr_state')['predictor'].mean()
state_counts = X_temp_data['addr_state'].value_counts()
means_smoothed = ((state_counts*state_means)+(m*global_default_mean))/(state_counts+m)

X_train['state_enc'] = X_train['addr_state'].map(means_smoothed)
X_test['state_enc'] = X_test['addr_state'].map(means_smoothed)
X_test['state_enc'] = X_test['state_enc'].fillna(global_default_mean)
categorical_features.remove('addr_state')

X_encoded=pd.get_dummies(X_train[categorical_features],drop_first=True,sparse=False,dtype=int)
X_train = pd.concat([X_train[numerical_features],X_encoded],axis=1)

X_encoded_test = pd.get_dummies(X_test[categorical_features],drop_first=True,sparse=False,dtype=int)
X_test = pd.concat([X_test[numerical_features],X_encoded_test],axis=1)
#align columns
X_test = X_test.reindex(columns=X_train.columns,fill_value=0)




''''''

zero_cols = [
    'max_bal_bc', 'all_util', 'il_util', 'open_acc_6m', 
    'open_il_12m', 'open_il_24m', 'open_rv_12m', 'open_rv_24m', 'inq_last_12m',
    'open_act_il', 'total_bal_il', 'total_il_high_credit_limit', 'is_consolidation'
]

flag_cols = [
    'mths_since_last_delinq', 'mths_since_last_record', 
    'mths_since_recent_bc_dlq', 'mths_since_recent_revol_delinq', 
    'mths_since_recent_inq', 'mths_since_rcnt_il',
    'mths_since_last_major_derog'
]

median_cols = [
    'months_sincefrst_credit', 'annual_inc', 'inq_last_6mths', 
    'revol_util', 'total_acc', 'pub_rec', 'open_acc', 
    'mo_sin_old_rev_tl_op', 'num_rev_accts', 'tot_hi_cred_lim',
    'acc_open_past_24mths', 'num_bc_sats', 'num_sats', 'mort_acc',
    'mths_since_recent_bc', 'total_bc_limit', 'pub_rec_bankruptcies',
    'total_rev_hi_lim', 'inq_fi', 'avg_cur_bal', 'bc_open_to_buy', 
    'bc_util', 'mo_sin_old_il_acct', 'mo_sin_rcnt_rev_tl_op', 
    'mo_sin_rcnt_tl', 'num_accts_ever_120_pd', 'num_actv_bc_tl', 
    'num_actv_rev_tl', 'num_bc_tl', 'num_il_tl', 'num_op_rev_tl', 
    'num_rev_tl_bal_gt_0', 'num_tl_90g_dpd_24m', 'num_tl_op_past_12m', 
    'pct_tl_nvr_dlq', 'percent_bc_gt_75',
    'total_cu_tl', 'total_bal_ex_mort', 'num_tl_30dpd', 'num_tl_120dpd_2m', 'chargeoff_within_12_mths'
]

set_config(transform_output="pandas")

preprocessing = ColumnTransformer([
    ('zeros',SimpleImputer(strategy='constant',fill_value=0),zero_cols),
    ('flags',SimpleImputer(strategy= 'median',add_indicator=True),flag_cols),
    ('median',SimpleImputer(strategy='median'),median_cols)
],remainder='passthrough')

X_train = preprocessing.fit_transform(X_train)
X_test = preprocessing.transform(X_test)





"""Feature Engineering"""

eps = 0.001 # to avoid dividing by zero
# 1. Loan-to-Income
X_train['FE_loan_to_income'] = X_train['remainder__loan_amnt'] / (X_train['median__annual_inc'] + eps)
X_test['FE_loan_to_income'] = X_test['remainder__loan_amnt'] / (X_test['median__annual_inc'] + eps)

# 2. Monthly Free Cash Flow
X_train['FE_free_cash_flow'] = (X_train['median__annual_inc'] / 12) * (1 - (X_train['remainder__dti'] / 100))
X_test['FE_free_cash_flow'] = (X_test['median__annual_inc'] / 12) * (1 - (X_test['remainder__dti'] / 100))

# 3. Credit Line Activity Ratio
X_train['FE_activity_ratio'] = X_train['median__num_actv_rev_tl'] / (X_train['median__num_op_rev_tl'] + eps)
X_test['FE_activity_ratio'] = X_test['median__num_actv_rev_tl'] / (X_test['median__num_op_rev_tl'] + eps)

MI/Correlation/ VIF 

In [ ]:
pd.set_option('display.max_columns', None)  
pd.set_option('display.width', 200)  
pd.set_option('display.expand_frame_repr', False) 


mi = mutual_info_classif(X_train,y_train,random_state=11)


vmc= pd.DataFrame({
    'feature':X_train.columns,
    'Mutual Information': mi
})



vmc = vmc.sort_values(by='Mutual Information',ascending=False).reset_index(drop=True)
print(vmc)


Winsorize

In [ ]:
upperbounds = X_train.quantile(0.99)
lowerbounds = X_train.quantile(.01)

X_train = X_train.clip(lower=lowerbounds,upper=upperbounds,axis=1)
X_test = X_test.clip(lower=lowerbounds,upper=upperbounds,axis=1)

Cohen's D and Confidence intervals -- manual calculation

In [ ]:
pd.set_option('display.max_columns', None)  
pd.set_option('display.width', 200)  
pd.set_option('display.expand_frame_repr', False)

def ci_cohen(X,y):
    df = X.copy()

    df['predictor'] = y

    group_one = df[df['predictor']==1]
    group_zero = df[df['predictor']==0]

    features = X.columns

    results = []
    for i in features:
        g_1 = group_one[i]
        g_0 = group_zero[i]

        n_1 ,n_0 = len(g_1),len(g_0)
        mean_1,mean_0= np.mean(g_1),np.mean(g_0)
        var_1,var_0 = np.var(g_1,ddof=1),np.var(g_0,ddof=1)


        s_pooled = np.sqrt(((n_1-1)* var_1+(n_0-1)*var_0)
                           /(n_1+n_0-2))
        
        d= (mean_1- mean_0)/(s_pooled)

        se_diff = np.sqrt((var_1/n_1)+(var_0/n_0))
        margin_of_error = 1.96*se_diff

        diff_means = mean_1-mean_0
        ci_lower = diff_means - margin_of_error
        ci_upper = diff_means + margin_of_error

        results.append({
            'Feature':i,
            'Cohens d':abs(round(d,2)),
            'CI, (lower,Upper)':f'{round(ci_lower,2)}, {round(ci_upper,2)}',
            'mean difference (default-paid)': round(diff_means,2)
        })
    
    results_df = pd.DataFrame(results).sort_values(by='Cohens d',ascending=False).reset_index(drop=True)

    return(results_df)


cohen_ci = ci_cohen(X_train,y_train)
print(cohen_ci)

Model + cv  for roc auc 

In [ ]:
logreg = Pipeline([
    ('scaler',StandardScaler()),
    ('model',LogisticRegression(max_iter=1000,random_state=11,class_weight='balanced'))
])

cvr = cross_val_score(logreg,
                      X_train,
                      y_train,
                      cv=5,
                      scoring='roc_auc',
                      n_jobs=-1)

print(cvr)
print(f'Mean {cvr.mean()}')
print(f'Std. {cvr.std()}')

Hypertune our Logistic Regression using optuna:

In [ ]:
def objective_log(trial):

    penalty = trial.suggest_categorical('penalty', ['l1', 'l2', 'elasticnet', None])
    
    if penalty == 'l1':
        solver = trial.suggest_categorical('solver_l1', ['liblinear', 'saga'])
    elif penalty == 'elasticnet':
        solver = 'saga'  
    elif penalty == 'l2':
        solver = trial.suggest_categorical('solver_l2', ['lbfgs', 'newton-cg', 'saga', 'liblinear'])
    else:
        solver = trial.suggest_categorical('solver_none', ['lbfgs', 'newton-cg', 'saga'])

    params = {
        'C': trial.suggest_float('C', 1e-4, 100.0, log=True),
        'penalty': penalty,
        'solver': solver,
        'max_iter': trial.suggest_int('max_iter', 500, 2000), # Increased for convergence
        'tol': trial.suggest_float('tol', 1e-5, 1e-2, log=True),
    }
    
    if penalty == 'elasticnet':
        params['l1_ratio'] = trial.suggest_float('l1_ratio', 0.0, 1.0)
    
    model = Pipeline([
        ('scaler', StandardScaler()),
        ('LinearR', LogisticRegression(**params, random_state=11,class_weight='balanced'))
    ])

    scores = cross_val_score(
        model,
        X_train,
        y_train,
        cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=11),
        scoring='roc_auc',
        n_jobs=-1
    )

    return scores.mean()

study_log = optuna.create_study(direction='maximize')
study_log.optimize(objective_log,n_trials=20)

print(f'Best AUC: {study_log.best_value}')
print(f'Best Params')

for k,v in study_log.best_params.items():
    print(f'{k}:{v}')


Learning Curve

In [ ]:
def plot_learning_curve(estimator,X,y):
    train_sizes,train_scores,test_scores = learning_curve(
        estimator,X,y,cv=5,n_jobs=-1,
        train_sizes=np.linspace(0.1,1,10),
        scoring='roc_auc'
    )
    train_mean = np.mean(train_scores,axis=1)
    train_std = np.std(train_scores,axis=1)
    test_mean = np.mean(test_scores,axis=1)
    test_std = np.std(test_scores,axis=1)

    plt.plot(train_sizes,train_mean,label='Training Score',color='#d65f5f',marker='o')
    plt.fill_between(train_sizes,train_mean - train_std,train_mean+train_std,alpha=0.15,color='#d65f5f')

    plt.plot(train_sizes,test_mean,label='Cross Validation score',color='#4878d0', marker='o')
    plt.fill_between(train_sizes, test_mean - test_std, test_mean + test_std, alpha=0.15, color='#4878d0')

    plt.title("Learning Curve for Logistic Regression")
    plt.xlabel('Training Samples')
    plt.ylabel('AUC Score')
    plt.legend()
    plt.grid(True,linestyle='--', alpha=0.7)
    plt.savefig('Images_log/LearningCurve_Log.png',dpi=300,bbox_inches='tight')
    plt.show()


refined_params_model= Pipeline([
    ('scaler', StandardScaler()),
    ('LinearR', LogisticRegression(C=0.18524697655221053,max_iter=1991,tol=0.005427682703976469,
                                   solver='saga',penalty='l1',random_state=11,class_weight='balanced'))
    ])

plot_learning_curve(refined_params_model,X_train,y_train)
#std/mean --> cv=5

weights -> plots

In [ ]:
refined_params_model= Pipeline([
    ('scaler', StandardScaler()),
    ('LinearR', LogisticRegression(C=0.18524697655221053,max_iter=1991,tol=0.005427682703976469,
                                   solver='saga',penalty='l1',random_state=11,class_weight='balanced'))
    ])

refined_params_model.fit(X_train,y_train)
model  = refined_params_model.named_steps['LinearR']
coefs = model.coef_[0]
odds_ratios = np.exp(coefs)


features=X_train.columns

model_results = pd.DataFrame({
    'Feature':features,
    'Coef':coefs,
    'odds':odds_ratios
})
mr = model_results.sort_values(by='odds').reset_index(drop=True)
print(mr)

Weight Plot

In [ ]:
plt.figure(figsize=(14,12))
colors = ['#d65f5f' if c>0 else '#4878d0' for c in mr['Coef']]

plt.barh(mr['Feature'], mr['Coef'],color=colors)
plt.axvline(x=0,color='black',linestyle='--',linewidth=2)
plt.title('LogisticRegression feature weights',fontsize=14,pad=15)
plt.xlabel('Impact on Default',fontsize=12)
plt.savefig('Images_log/LogisticRegression_feature_weight.png',dpi=300,bbox_inches='tight')

plt.tight_layout()
plt.show()



In [ ]:
mr_abs = mr.loc[:,['Feature','Coef','odds']]
mr_abs['abs Coef'] = mr_abs['Coef'].abs()
mr_abs=mr_abs.sort_values(by='abs Coef',ascending=False)
mr_abs=mr_abs.drop('Coef',axis=1)
print(mr_abs.head(10))

Effect plot

In [ ]:
params = {
    'C':90.31097151134563,
    'solver':'saga',
    'max_iter':1595,
    'tol':0.009993372798221852,
    'l1_ratio':0.6871072356705394
}

model = Pipeline([
        ('scaler', StandardScaler()),
        ('LinearR', LogisticRegression(**params, random_state=11,class_weight='balanced'))
    ])

model.fit(X_train,y_train)

#repeating because different columns before.
scaler = model.named_steps['scaler']
X_train_scaled = scaler.transform(X_train)
X_train_scaled_df = pd.DataFrame(X_train_scaled,columns=X_train.columns)
weights = model.named_steps['LinearR'].coef_[0]


effects_df = X_train_scaled_df * weights

means_abs_effects = effects_df.abs().mean().sort_values(ascending=False)

plt.figure(figsize=(12,8))
sns.boxplot(
    data=effects_df,
    orient='h',
    palette='vlag',
    showfliers=False, #take out outliers
)

plt.axvline(x=0,color='red',linestyle='--',linewidth=1.5,alpha=0.7)
plt.title('Feature effects')
plt.xlabel('Effects on Log-Odds')
plt.grid(axis='x',linestyle='--',alpha=0.5)

#plt.savefig('Images_log/Logreg_effectplot.png',dpi=300,bbox_inches='tight')
plt.show()